# 📉 Cuaderno 03 — Mínimos cuadrados lineal y cuadrático

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

Cuando los datos traen ruido, no se interpola: se ajusta. Aquí se construye el criterio de mínimos cuadrados, se resuelven las ecuaciones normales y se aprende a comparar modelos con $s_{y/x}$ y $r^2$.

> 📖 Teoría: [Mínimos cuadrados](../docs/metodos/04_minimos_cuadrados_lineal_cuadratico.md)

In [ ]:
#@title ⚙️ Entorno (ejecuta PRIMERO)
import os, sys, importlib

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _buscar(nombre, saltos=6):
    """Busca un archivo junto al cuaderno, en la carpeta actual y en lib/ de los padres."""
    bases = [os.getcwd()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    if ruta_nb:
        bases.append(os.path.dirname(os.path.abspath(ruta_nb)))
    for base in list(bases):
        b = base
        for _ in range(saltos):
            for cand in (os.path.join(b, nombre), os.path.join(b, "lib", nombre)):
                if os.path.exists(cand):
                    return os.path.abspath(cand)
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


rutas = {n: _buscar(n) for n in ("robomat.py", "ajuste_curvas.py")}
faltan = [n for n, r in rutas.items() if r is None]
if faltan:
    raise FileNotFoundError(
        "No encontré " + ", ".join(faltan) +
        ". En Colab, sube la carpeta lib/ junto al cuaderno.")

_carpeta_lib = os.path.dirname(rutas["robomat.py"])
if _carpeta_lib not in sys.path:
    sys.path.insert(0, _carpeta_lib)

import robomat as _robomat
importlib.reload(_robomat)
from robomat import *               # noqa: F401,F403

import ajuste_curvas as ac
importlib.reload(ac)

import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatSlider
except ImportError:
    interact = None

print("✅ Entorno cargado desde:", _carpeta_lib)
print("✅ Entorno:", "Google Colab" if ES_COLAB else "local")

### 🎯 Objetivos

- Explicar por qué se minimiza la suma de **cuadrados** y no otra cosa.
- Obtener la recta de regresión con las ecuaciones normales (17.4)–(17.7).
- Calcular $S_r$, $S_t$, $r^2$, $r$ y $s_{y/x}$.
- Ajustar un polinomio cuadrático con (17.19)–(17.20).
- Ajustar una sinusoide con las ecuaciones normales (19.11)–(19.12).

**Referencia del libro:** secciones 17.1, 17.2 y 19.1.

## 1. ¿Por qué cuadrados?

Con datos de la tabla 17.1, probamos muchas pendientes y graficamos $S_r$.
Si el criterio es correcto, debe haber **un solo** mínimo.

In [ ]:
x = np.arange(1.0, 8.0)
y = np.array([0.5, 2.5, 2.0, 4.0, 3.5, 6.0, 5.5])

def Sr(a0, a1, x, y):
    return float(np.sum((y - a0 - a1 * x) ** 2))

a1_grid = np.linspace(0.2, 1.4, 200)
a0_grid = np.mean(y) - a1_grid * np.mean(x)     # ecuación (17.7)
plt.figure(figsize=(8, 4.5))
plt.plot(a1_grid, [Sr(a0, a1, x, y) for a0, a1 in zip(a0_grid, a1_grid)])
plt.xlabel("pendiente $a_1$"); plt.ylabel("$S_r$"); plt.grid(alpha=0.3)
plt.title("$S_r$ es cuadrática: un único mínimo")
plt.show()

## 2. Regresión lineal

Las ecuaciones normales son

$$n a_0+\left(\sum x_i\right)a_1=\sum y_i,\qquad
\left(\sum x_i\right)a_0+\left(\sum x_i^2\right)a_1=\sum x_i y_i, \tag{17.4–17.5}$$

y su solución es

$$a_1=\frac{n\sum x_i y_i-\sum x_i\sum y_i}{n\sum x_i^2-(\sum x_i)^2},\qquad
a_0=\bar y-a_1\bar x. \tag{17.6–17.7}$$

In [ ]:
ajuste = ac.minimos_cuadrados(x, y, 1)
a0, a1 = ajuste["coef"]
print(f"y = {a0:.8f} + {a1:.8f} x")
print(f"Sr   = {ajuste['Sr']:.6f}")
print(f"St   = {ajuste['St']:.6f}")
print(f"syx  = {ajuste['syx']:.6f}")
print(f"r2   = {ajuste['r2']:.6f}")
print(f"r    = {np.sqrt(ajuste['r2']):.6f}")
print("pasa por (x̄, ȳ):", np.isclose(a0 + a1 * x.mean(), y.mean()))
print("residual medio:", np.mean(ajuste["residuales"]))

### Las ecuaciones normales, con sintaxis del curso

El mismo ajuste se obtiene resolviendo (17.4)–(17.5) como un sistema lineal.
En la biblioteca del curso las matrices se escriben con `M.sym[[...]]`
(exacto) y se muestran con `mostrar()`; el resultado debe coincidir con
`ac.minimos_cuadrados`.

In [ ]:
n = len(x)
A = M.sym[[n, sum(x)], [sum(x), sum(x**2)]]
b = M.sym[[sum(y)], [sum(x * y)]]
mostrar(A)
mostrar(b)

sol = A.LUsolve(b)
mostrar(sol)
print("coincide con ac.minimos_cuadrados:",
      verificar(sol, M.sym[[float(a0)], [float(a1)]]))

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(x, y, "o", ms=8, label="datos")
plt.plot(x, ajuste["pred"], "-", label=f"y = {a0:.4f} + {a1:.4f} x")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Regresión lineal, ejemplo 17.1")
plt.show()

plt.figure(figsize=(8, 3.5))
plt.stem(x, ajuste["residuales"], basefmt=" ")
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("x"); plt.ylabel("residuo $e_i$"); plt.grid(alpha=0.3)
plt.title("Residuales: deben verse sin patrón")
plt.show()

## 3. Regresión cuadrática

Para $y=a_0+a_1x+a_2x^2$ las ecuaciones normales son

$$\begin{aligned}
(n)a_0+\left(\sum x_i\right)a_1+\left(\sum x_i^2\right)a_2&=\sum y_i,\\
\left(\sum x_i\right)a_0+\left(\sum x_i^2\right)a_1+\left(\sum x_i^3\right)a_2&=\sum x_iy_i,\\
\left(\sum x_i^2\right)a_0+\left(\sum x_i^3\right)a_1+\left(\sum x_i^4\right)a_2&=\sum x_i^2y_i.
\end{aligned} \tag{17.19}$$

El error estándar pierde $m+1$ grados de libertad:

$$s_{y/x}=\sqrt{\frac{S_r}{n-(m+1)}}. \tag{17.20}$$

In [ ]:
x2 = np.arange(0.0, 6.0)
y2 = np.array([2.1, 7.7, 13.6, 27.2, 40.9, 61.1])

cuad = ac.minimos_cuadrados(x2, y2, 2)
print("coeficientes [a0, a1, a2] =", np.round(cuad["coef"], 5))
print(f"Sr  = {cuad['Sr']:.5f}")
print(f"syx = {cuad['syx']:.4f}")
print(f"r2  = {cuad['r2']:.5f}")

Para el polinomio cuadrático el sistema es de $3\times3$, exactamente (17.19):

$$\begin{bmatrix}
n & \sum x_i & \sum x_i^2\\
\sum x_i & \sum x_i^2 & \sum x_i^3\\
\sum x_i^2 & \sum x_i^3 & \sum x_i^4
\end{bmatrix}
\begin{bmatrix}a_0\\a_1\\a_2\end{bmatrix}
=\begin{bmatrix}\sum y_i\\ \sum x_i y_i\\ \sum x_i^2 y_i\end{bmatrix}.$$

In [ ]:
n2 = len(x2)
A2 = M.sym[[n2, sum(x2), sum(x2**2)],
           [sum(x2), sum(x2**2), sum(x2**3)],
           [sum(x2**2), sum(x2**3), sum(x2**4)]]
b2 = M.sym[[sum(y2)], [sum(x2 * y2)], [sum(x2**2 * y2)]]
mostrar(A2)
mostrar(b2)

sol2 = A2.LUsolve(b2)
mostrar(sol2)
esperado2 = M.sym[[float(cuad["coef"][0])],
                  [float(cuad["coef"][1])],
                  [float(cuad["coef"][2])]]
print("coincide con ac.minimos_cuadrados:", verificar(sol2, esperado2))

In [ ]:
xs = np.linspace(-0.2, 5.2, 200)
plt.figure(figsize=(8, 4.5))
plt.plot(x2, y2, "o", ms=8, label="datos")
plt.plot(xs, ac.evaluar_polinomio(cuad["coef"], xs), "-", label="cuadrática")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Regresión cuadrática, ejemplo 17.5")
plt.show()

## 4. Laboratorio: ¿qué grado elegir?

Subir el grado siempre reduce $S_r$. La pregunta correcta es si el modelo
mejora la **predicción** o sólo copia el ruido. Comparamos grados 1 a 5 sobre
los mismos datos.

In [ ]:
print(f"{'grado':>5} {'Sr':>10} {'syx':>8} {'r2':>8}")
for m in range(1, 6):
    r = ac.minimos_cuadrados(x2, y2, m)
    print(f"{m:>5} {r['Sr']:>10.6f} {r['syx']:>8.4f} {r['r2']:>8.5f}")

print("\nOjo: con n=6, el grado 5 interpola exactamente los datos (Sr ≈ 0),")
print("pero no por eso es un mejor modelo.")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(x2, y2, "o", ms=8, label="datos", zorder=5)
for m, estilo in [(1, ":"), (2, "-"), (5, "--")]:
    r = ac.minimos_cuadrados(x2, y2, m)
    plt.plot(xs, ac.evaluar_polinomio(r["coef"], xs), estilo, label=f"grado {m}")
plt.ylim(-10, 80); plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Más grado no siempre es mejor")
plt.show()

## 5. Ajuste sinusoidal (sección 19.1)

La sinusoide

$$f(t)=A_0+A_1\cos(\omega_0t)+B_1\sin(\omega_0t) \tag{19.6}$$

es un modelo **lineal en los coeficientes**. Con datos igualmente espaciados
que completan un periodo, las ecuaciones normales se simplifican a

$$A_0=\frac{\sum y}{N},\qquad
A_1=\frac{2}{N}\sum y\cos(\omega_0t),\qquad
B_1=\frac{2}{N}\sum y\sin(\omega_0t). \tag{19.14–19.16}$$

Usamos los datos del ejemplo 19.1: $y=1.7+\cos(4.189t+1.0472)$.

In [ ]:
t = np.arange(0, 1.36, 0.15)
ys = np.array([2.200, 1.595, 1.031, 0.722, 0.786,
               1.200, 1.805, 2.369, 2.678, 2.614])
omega0 = 4.189

sen = ac.ajuste_sinusoidal(t, ys, omega0)
print(f"A0 = {sen['A0']:.4f}")
print(f"A1 = {sen['A1']:.4f}")
print(f"B1 = {sen['B1']:.4f}")
print(f"C1 = {sen['C1']:.4f}")
print(f"theta = {sen['theta']:.4f} rad")
print(f"Sr = {sen['Sr']:.3e}")

tt = np.linspace(t.min(), t.max(), 300)
plt.figure(figsize=(8, 4.5))
plt.plot(t, ys, "o", ms=8, label="datos")
plt.plot(tt, sen["A0"] + sen["A1"]*np.cos(omega0*tt) + sen["B1"]*np.sin(omega0*tt),
         "-", label="ajuste (19.6)")
plt.axhline(sen["A0"], color="0.6", ls=":")
plt.xlabel("t"); plt.ylabel("y"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Ajuste por mínimos cuadrados de una sinusoide")
plt.show()

**Nota fina.** El libro reporta $A_1=0.500$, $B_1=-0.866$ usando las fórmulas
simplificadas (19.14–19.16). Aquellas fórmulas suponen un número entero de
periodos; aquí la ventana cubre $0.9$ periodos. Por eso la solución exacta de
las ecuaciones normales (19.12) difiere en la cuarta decimal. Ambas son
correctas **bajo sus supuestos**; hay que declarar cuál se usó.

## 6. Autoevaluación

**Pregunta 1.** Para la tabla 17.1, ¿cuánto vale la pendiente $a_1$?

In [ ]:
respuesta_1 = 0.0     # <-- cambia este valor

correcta_1 = ac.minimos_cuadrados(np.arange(1.0, 8.0),
                                 np.array([0.5, 2.5, 2.0, 4.0, 3.5, 6.0, 5.5]),
                                 1)["coef"][1]
print("✅ Correcto" if verificar(respuesta_1, correcta_1, tol=1e-4)
      else f"❌ Revisa la ecuación (17.6). Valor esperado: {correcta_1:.7f}")

**Pregunta 2.** ¿Cuál es el error estándar $s_{y/x}$ de la regresión lineal
de la tabla 17.1?

In [ ]:
respuesta_2 = 0.0     # <-- cambia este valor

correcta_2 = ac.minimos_cuadrados(np.arange(1.0, 8.0),
                                 np.array([0.5, 2.5, 2.0, 4.0, 3.5, 6.0, 5.5]),
                                 1)["syx"]
print("✅ Correcto" if verificar(respuesta_2, correcta_2, tol=1e-3)
      else f"❌ Revisa la ecuación (17.9). Valor esperado: {correcta_2:.6f}")

**Pregunta 3 (decisión).** Un compañero reporta $r^2=0.99$ para un ajuste
lineal de datos que claramente forman una curva. ¿Qué le pides antes de
aceptar el modelo? Escribe tu respuesta.

In [ ]:
respuesta_3 = '''
Escribe aquí tu respuesta (piensa en la gráfica y en los residuales).
'''
print(respuesta_3)

## 7. Resumen

- El criterio (17.3) minimiza la suma de cuadrados porque produce un mínimo
  único.
- Las ecuaciones normales (17.4)–(17.5) y (17.19) son sistemas lineales en
  los coeficientes.
- $s_{y/x}$ mide la dispersión alrededor del modelo y ajusta por grados de
  libertad.
- $r^2$ es útil para comparar, pero **no** decide el grado ni la forma.
- El ajuste sinusoidal es el mismo criterio con otra base de funciones.